# Hybrid LSTM+GAT Fraud Detection — IBM & Sparkov
Standalone companion to *Isolating Graph Topology from Model Architecture in
GNN-Based Fraud Detection* (Amiri & Jaf, University of Sunderland, 2026).
Extends the fixed-architecture, variable-topology framework with three
architecture variants per dataset, holding topology fixed and varying
architecture instead — the inverse experiment.

| File | Architecture | Node granularity |
|---|---|---|
| `<dataset>/lstm_gat_sequential_model.py` | LSTM → GAT pipeline | Transaction |
| `<dataset>/lstm_gat_parallel_model.py` | LSTM ‖ GAT, cross-attention fusion | Transaction |
| `<dataset>/account_gat_homogeneous_model.py` | Unchanged GATv2, new topology | Account |

Run the **IBM** section, the **Sparkov** section, or both — they're
independent and don't share state.

---
### Before you start
1. **Runtime → Change runtime type → T4 GPU** (free tier)
2. Update `REPO_URL` in Cell 2
3. Skip whichever dataset section you don't need

---
## Cell 1 — Install dependencies

In [ ]:
import subprocess, sys
import torch

torch_version = torch.__version__.split('+')[0]
cuda_version  = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch_version}  |  CUDA: {cuda_version}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'scikit-learn', 'pandas', 'matplotlib', 'pyarrow'], check=True)
print('Done.')

---
## Cell 2 — Clone this repo

In [ ]:
import os

REPO_URL = 'https://github.com/<your-username>/hybrid-gnn-lstm-fraud.git'  # ← update

if not os.path.isdir('/content/hybrid-gnn-lstm-fraud'):
    !git clone -q {REPO_URL} /content/hybrid-gnn-lstm-fraud

os.makedirs('/content/outcomes', exist_ok=True)
print('Repo ready at /content/hybrid-gnn-lstm-fraud')

---
## Cell 2b — (Optional) Mount Google Drive
Only needed if your dataset files live in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
# Part A — IBM Dataset

## Cell 3 — Load and preprocess
Set `IBM_DATA_PATH` to your actual `reduced_dataset.parquet` (or
equivalent) location before running.

In [ ]:
import os, sys

IBM_DATA_PATH = '/content/drive/MyDrive/reduced_dataset.parquet'  # ← update

os.chdir('/content/hybrid-gnn-lstm-fraud/ibm')
sys.path.insert(0, os.getcwd())

import config as ibm_cfg
cfg_ibm = ibm_cfg.IBMFraudConfig()
cfg_ibm.OUTCOME_DIR = '/content/outcomes/ibm'

import utils as ibm_utils
df_ibm = ibm_utils.load_and_preprocess(path=IBM_DATA_PATH, cfg=cfg_ibm)
print(f'{len(df_ibm):,} transactions loaded.')

---
## Cell 4 — IBM: GATv2 baseline (fixed architecture, for comparison)
~25–40 min on T4 GPU (all 3 strategies).

In [ ]:
import gatv2_model as ibm_gatv2

gatv2_results_ibm = ibm_gatv2.run_all_strategies(df_ibm, cfg_ibm)
print('\nIBM GATv2 baseline done.')

---
## Cell 5 — IBM: Sequential LSTM→GAT
~35–55 min on T4 GPU.

In [ ]:
import lstm_gat_sequential_model as ibm_lstm_seq

lstm_seq_results_ibm = ibm_lstm_seq.run_all_strategies(df_ibm, cfg_ibm)
print('\nIBM Sequential LSTM→GAT done.')

---
## Cell 6 — IBM: Parallel LSTM‖GAT
~45–70 min on T4 GPU.

In [ ]:
import lstm_gat_parallel_model as ibm_lstm_par

lstm_par_results_ibm = ibm_lstm_par.run_all_strategies(df_ibm, cfg_ibm)
print('\nIBM Parallel LSTM‖GAT done.')

---
## Cell 7 — IBM: Homogeneous Account-Level GAT
~5–10 min (far fewer nodes than the transaction-level graphs).

In [ ]:
import account_gat_homogeneous_model as ibm_acct_gat

acct_cfg_ibm = ibm_acct_gat.AccountFraudConfig()
acct_cfg_ibm.OUTCOME_DIR = '/content/outcomes/ibm'

df_accounts_ibm = ibm_acct_gat.build_account_dataframe(df_ibm, acct_cfg_ibm)
acct_results_ibm = ibm_acct_gat.run_all_strategies(df_accounts_ibm, acct_cfg_ibm)
print('\nIBM Homogeneous Account-Level GAT done.')

---
## Cell 8 — IBM: combined comparison table

In [ ]:
import pandas as pd

all_results_ibm = {}
all_results_ibm.update(gatv2_results_ibm)
all_results_ibm.update(lstm_seq_results_ibm)
all_results_ibm.update(lstm_par_results_ibm)
all_results_ibm.update(acct_results_ibm)

rows = []
for key, res in all_results_ibm.items():
    m = res['test_metrics']
    rows.append({'run': key, 'model_arch': res.get('model_arch', ''),
                 'graph_strategy': res.get('graph_strategy', ''),
                 'f1': round(m['f1'], 4), 'prec': round(m['prec'], 4), 'rec': round(m['rec'], 4),
                 'auc': round(m['auc'], 4), 'ap': round(m['ap'], 4)})

df_summary_ibm = pd.DataFrame(rows).sort_values(['model_arch', 'f1'], ascending=[True, False])
os.makedirs('/content/outcomes/ibm', exist_ok=True)
df_summary_ibm.to_csv('/content/outcomes/ibm/all_architectures_summary.csv', index=False)
df_summary_ibm

---
# Part B — Sparkov Dataset

## Cell 9 — Load and preprocess
Set `SPARKOV_TRAIN_PATH` / `SPARKOV_TEST_PATH` to your actual
`fraudTrain.csv` / `fraudTest.csv` location before running.
Independent of Part A — safe to run this section on its own.

In [ ]:
import os, sys

SPARKOV_TRAIN_PATH = '/content/fraudTrain.csv'  # ← update if needed
SPARKOV_TEST_PATH  = '/content/fraudTest.csv'   # ← update if needed

os.chdir('/content/hybrid-gnn-lstm-fraud/sparkov')
sys.path.insert(0, os.getcwd())

import config as sparkov_cfg
cfg_sparkov = sparkov_cfg.CardFraudConfig()
cfg_sparkov.OUTCOME_DIR = '/content/outcomes/sparkov'

import utils as sparkov_utils
df_sparkov = sparkov_utils.load_and_preprocess(
    train_path=SPARKOV_TRAIN_PATH, test_path=SPARKOV_TEST_PATH, cfg=cfg_sparkov)
print(f'{len(df_sparkov):,} transactions loaded.')

---
## Cell 10 — Sparkov: GATv2 baseline
~20–35 min on T4 GPU.

In [ ]:
import gatv2_model as sparkov_gatv2

gatv2_results_sparkov = sparkov_gatv2.run_all_strategies(df_sparkov, cfg_sparkov)
print('\nSparkov GATv2 baseline done.')

---
## Cell 11 — Sparkov: Sequential LSTM→GAT
~30–50 min on T4 GPU.

In [ ]:
import lstm_gat_sequential_model as sparkov_lstm_seq

lstm_seq_results_sparkov = sparkov_lstm_seq.run_all_strategies(df_sparkov, cfg_sparkov)
print('\nSparkov Sequential LSTM→GAT done.')

---
## Cell 12 — Sparkov: Parallel LSTM‖GAT
~40–65 min on T4 GPU.

In [ ]:
import lstm_gat_parallel_model as sparkov_lstm_par

lstm_par_results_sparkov = sparkov_lstm_par.run_all_strategies(df_sparkov, cfg_sparkov)
print('\nSparkov Parallel LSTM‖GAT done.')

---
## Cell 13 — Sparkov: Homogeneous Account-Level GAT
~5–10 min.

In [ ]:
import account_gat_homogeneous_model as sparkov_acct_gat

acct_cfg_sparkov = sparkov_acct_gat.CardAccountConfig()
acct_cfg_sparkov.OUTCOME_DIR = '/content/outcomes/sparkov'

df_accounts_sparkov = sparkov_acct_gat.build_account_dataframe(df_sparkov, acct_cfg_sparkov)
acct_results_sparkov = sparkov_acct_gat.run_all_strategies(df_accounts_sparkov, acct_cfg_sparkov)
print('\nSparkov Homogeneous Account-Level GAT done.')

---
## Cell 14 — Sparkov: combined comparison table

In [ ]:
all_results_sparkov = {}
all_results_sparkov.update(gatv2_results_sparkov)
all_results_sparkov.update(lstm_seq_results_sparkov)
all_results_sparkov.update(lstm_par_results_sparkov)
all_results_sparkov.update(acct_results_sparkov)

rows = []
for key, res in all_results_sparkov.items():
    m = res['test_metrics']
    rows.append({'run': key, 'model_arch': res.get('model_arch', ''),
                 'graph_strategy': res.get('graph_strategy', ''),
                 'f1': round(m['f1'], 4), 'prec': round(m['prec'], 4), 'rec': round(m['rec'], 4),
                 'auc': round(m['auc'], 4), 'ap': round(m['ap'], 4)})

df_summary_sparkov = pd.DataFrame(rows).sort_values(['model_arch', 'f1'], ascending=[True, False])
os.makedirs('/content/outcomes/sparkov', exist_ok=True)
df_summary_sparkov.to_csv('/content/outcomes/sparkov/all_architectures_summary.csv', index=False)
df_summary_sparkov

---
## Cell 15 — Download everything

In [ ]:
!cd /content/outcomes && zip -qr /content/outcomes.zip .
from google.colab import files
files.download('/content/outcomes.zip')